# Lesson 3: Agentic Search

**Student:** Tanish Gheewala  
**Purpose:** Compare conventional search and webpage scraping with Tavily's agent-oriented search results.  
**Completion date:** Recorded by the final executed cell in UTC.

Adapted from the supplied Lesson_3_Student notebook. Dependencies are installed by the Docker execution command. API keys are loaded from the environment and are never printed. Live results are generated when this notebook is executed.

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os
from tavily import TavilyClient

load_dotenv(Path.cwd() / ".env", override=False)
if not os.environ.get("TAVILY_API_KEY"):
    raise RuntimeError("Set TAVILY_API_KEY in your local .env before running.")
client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
print("Tavily client initialized; API key is not displayed.")

In [ ]:
result = client.search("What is in Nvidia's new Blackwell GPU?", include_answer=True, max_results=3)
print(result.get("answer") or "No synthesized answer returned; see sources below.")
for item in result.get("results", []):
    print(item["title"], "-", item["url"])
assert result.get("results"), "Search returned no source results."

## Regular search

In [ ]:
city = "San Diego"
query = f"What is the current weather forecast in {city}, California? Should I travel there today?"
print(query)

The regular-search example uses DDGS to obtain URLs, then requests and parses a live webpage. If DDGS is unavailable, it explicitly falls back to fresh Tavily URL results. No canned search results or fabricated weather data are used.

In [ ]:
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import re

search_backend = "DDGS"

def search(query, max_results=3):
    global search_backend
    try:
        results = list(DDGS(timeout=10).text(query, max_results=max_results))
        urls = [item["href"] for item in results if item.get("href")]
        if urls:
            return urls
        raise RuntimeError("No DDGS results")
    except Exception as exc:
        search_backend = "Tavily URL fallback"
        print(f"DDGS unavailable ({type(exc).__name__}); fetching fresh URLs with Tavily.")
        fallback = client.search(query, max_results=max_results, include_answer=False)
        return [item["url"] for item in fallback.get("results", [])]

regular_urls = search(query + " site:weather.gov")
print("Search backend:", search_backend)
for url in regular_urls:
    print(url)

In [ ]:
def scrape_weather_info(url):
    """Fetch a real page; return None on a network error or blocked response."""
    try:
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        print(f"HTTP {response.status_code}: {url}")
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        if len(soup.get_text(" ", strip=True)) < 100:
            print("Page contained too little readable text.")
            return None
        return soup
    except requests.RequestException as exc:
        print(f"Could not retrieve page ({type(exc).__name__}).")
        return None

print("Webpage scraper defined with a 10-second request timeout.")

The next cell displays a short HTML excerpt from the first retrievable live result. If search results are blocked, the official San Diego National Weather Service office page is tried and explicitly identified as a fallback. Network failures remain visible in the output.

In [ ]:
soup = None
url = None
candidates = list(dict.fromkeys(regular_urls[:3] + ["https://www.weather.gov/sgx/"]))
for candidate in candidates:
    if candidate not in regular_urls:
        print("Trying the official San Diego NWS office page as a fallback.")
    page = scrape_weather_info(candidate)
    if page is not None:
        soup, url = page, candidate
        break
if soup is None:
    print("No live page could be retrieved. Regular scraping was not successful in this run.")
else:
    print(f"Website: {url}\n")
    print(str(soup.body or soup)[:3000])

In [ ]:
weather_data = ""
if soup is not None:
    sections = [tag.get_text(" ", strip=True) for tag in soup.find_all(["h1", "h2", "h3", "p", "li"])]
    weather_data = re.sub(r"\s+", " ", " ".join(sections)).strip()
    if not weather_data:
        weather_data = soup.get_text(" ", strip=True)
    print(f"Website: {url}\n")
    print(weather_data[:10000])
else:
    print("No extracted text: live webpage retrieval failed in the preceding step.")

## Agentic Search

In [ ]:
result = client.search(query, max_results=2, include_answer=True)
assert result.get("results"), "Tavily returned no results."
print("Answer:", result.get("answer") or "No synthesized answer returned.")
for item in result["results"]:
    print("\nSource:", item["url"])
    print(item.get("content", ""))

In [ ]:
import json

# Search content is prose, not necessarily JSON. Serialize the actual response dictionary.
formatted_json = json.dumps(result, indent=2, ensure_ascii=False)
parsed_json = json.loads(formatted_json)
print(formatted_json)
print("\nJSON round-trip successful:", isinstance(parsed_json, dict))

## Comparison

Regular search supplies links; the application must fetch pages and remove HTML itself. Sites can block automated requests. Tavily supplies extracted source content and can supply a synthesized answer in a structured response, reducing the parsing work needed by an agent. Its results still need source and freshness checks. The weather answer is live search output, not a guaranteed travel recommendation.

The starter's unavailable `google_sample.png` illustration is omitted. The outputs above contain the actual search and scraping evidence for this run.

In [ ]:
from datetime import datetime, timezone

print("Student: Tanish Gheewala")
print("Execution completed (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Regular search backend:", search_backend)
print("Regular webpage text retrieved:", bool(weather_data))
print("Tavily results returned:", len(result["results"]))
print("Structured response is valid JSON:", isinstance(parsed_json, dict))
if not weather_data:
    print("Limitation: webpage scraping failed; this step needs a retry for full evidence.")